# 03 — Feature Engineering

**Workstream:** Price Prediction · **Batch 2 (Day 2-3)** · **Status: IMPLEMENTED**

Builds calendar + lag + rolling features and the two targets (1-day, 7-day ahead), then does the
chronological train/val/test split. All logic lives in `ai/price_prediction/src/features.py` — this
notebook only calls it and inspects the result (leakage checklist: `docs/price/target-and-leakage.md`).

Uses a freshly-generated full synthetic grid (9000 rows, 180 days × 10 markets × 5 commodities) rather than
notebook 01/02's 500-row quick-look snapshot — lag/rolling features need real history per series to be
meaningful, and 5-14 days/series (what the quick look had) isn't enough. Re-run notebooks 01-02 with a
larger `n_rows` later if you want the full chain to use the same file end-to-end; functionally identical
data, this just skips regenerating it twice.

In [1]:
import sys
from pathlib import Path
_candidate = Path.cwd()
_root = None
for _p in [_candidate, *_candidate.parents]:
    if (_p / "ai" / "common" / "config.py").exists():
        _root = _p
        break
sys.path.insert(0, str(_root))
print("Repo root:", _root)

Repo root: /home/claude/AgriConnect


In [2]:
import pandas as pd
from ai.price_prediction.src.ingest import generate_synthetic_fixture
from ai.price_prediction.src.features import build_feature_matrix, chronological_split
from ai.common.io_utils import save_df
from ai.common.config import DATA_PROCESSED_DIR

df = generate_synthetic_fixture(n_rows=100000, seed=42)
df["arrival_date_parsed"] = pd.to_datetime(df["arrival_date"], format="%d/%m/%Y")
print(f"Base data: {df.shape}")

08:32:19 | WARNING  | ai.price_prediction.src.ingest | Generated 9000 SYNTHETIC mandi price rows - NOT real data. Do not use for reported metrics.


Base data: (9000, 14)


## 1. Build features + targets

In [3]:
feat_df, category_maps = build_feature_matrix(df)
new_cols = [c for c in feat_df.columns if c not in df.columns]
print(f"Feature matrix: {feat_df.shape}")
print(f"New columns ({len(new_cols)}):", new_cols)
feat_df[["state","market","commodity","arrival_date_parsed","modal_price","lag_1","lag_7","rolling_mean_7","target_1d","target_7d"]].head(10)

Feature matrix: (9000, 34)
New columns (20): ['day_of_week', 'day_of_month', 'month', 'week_of_year', 'is_weekend', 'lag_1', 'lag_3', 'lag_7', 'lag_14', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_14', 'rolling_std_14', 'target_1d', 'target_7d', 'state_code', 'market_code', 'commodity_code', 'variety_code', 'grade_code']


,state,market,commodity,arrival_date_parsed,modal_price,lag_1,lag_7,rolling_mean_7,target_1d,target_7d
1800,Andhra Pradesh,Guntur Mandi,Onion,2026-02-15,1686.92,NaN,NaN,NaN,1636.92,1771.28
1801,Andhra Pradesh,Guntur Mandi,Onion,2026-02-16,1636.92,1686.92,NaN,NaN,1528.80,1662.78
1802,Andhra Pradesh,Guntur Mandi,Onion,2026-02-17,1528.80,1636.92,NaN,NaN,1655.23,1618.99
1803,Andhra Pradesh,Guntur Mandi,Onion,2026-02-18,1655.23,1528.80,NaN,1617.546667,1565.17,1733.84
1804,Andhra Pradesh,Guntur Mandi,Onion,2026-02-19,1565.17,1655.23,NaN,1626.967500,1770.03,1681.06
1805,Andhra Pradesh,Guntur Mandi,Onion,2026-02-20,1770.03,1565.17,NaN,1614.608000,1644.36,1804.87
1806,Andhra Pradesh,Guntur Mandi,Onion,2026-02-21,1644.36,1770.03,NaN,1640.511667,1771.28,1622.14
1807,Andhra Pradesh,Guntur Mandi,Onion,2026-02-22,1771.28,1644.36,1686.92,1641.061429,1662.78,1741.25
1808,Andhra Pradesh,Guntur Mandi,Onion,2026-02-23,1662.78,1771.28,1636.92,1653.112857,1618.99,1803.59
1809,Andhra Pradesh,Guntur Mandi,Onion,2026-02-24,1618.99,1662.78,1528.80,1656.807143,1733.84,1747.03


## 2. Chronological split (leakage rule 5 — no shuffling, ever)

In [4]:
train, val, test = chronological_split(feat_df, val_days=21, test_days=14)
print(f"train={len(train)}  val={len(val)}  test={len(test)}")
print(f"train: {train['arrival_date_parsed'].min().date()} -> {train['arrival_date_parsed'].max().date()}")
print(f"val  : {val['arrival_date_parsed'].min().date()} -> {val['arrival_date_parsed'].max().date()}")
print(f"test : {test['arrival_date_parsed'].min().date()} -> {test['arrival_date_parsed'].max().date()}")
assert train["arrival_date_parsed"].max() < val["arrival_date_parsed"].min()
assert val["arrival_date_parsed"].max() < test["arrival_date_parsed"].min()
print("\nNo temporal overlap between splits: confirmed.")

train=7250  val=1050  test=700
train: 2026-02-15 -> 2026-07-09
val  : 2026-07-10 -> 2026-07-30
test : 2026-07-31 -> 2026-08-13

No temporal overlap between splits: confirmed.


## 3. Leakage self-check\n\n`lag_1` for a row must equal the previous row's `modal_price` in the *same* series — never a future row.

In [5]:
sample = feat_df[(feat_df.state=="Maharashtra") & (feat_df.market=="Pune Mandi") & (feat_df.commodity=="Onion")].sort_values("arrival_date_parsed")
ok = (sample["modal_price"].shift(1).dropna().values == sample["lag_1"].dropna().values).all()
print("lag_1 correctness check:", ok)
assert ok, "Leakage bug: lag_1 does not match the true previous-row price."


lag_1 correctness check: True


## 4. Save train/val/test splits\n\nFeeds `04_baseline_and_models.ipynb`.

In [6]:
for name, split_df in [("train", train), ("val", val), ("test", test)]:
    save_df(split_df, DATA_PROCESSED_DIR / f"mandi_features_{name}_synthetic.csv")
import json as _json
with open(DATA_PROCESSED_DIR / "mandi_category_maps_synthetic.json", "w") as f:
    _json.dump(category_maps, f, indent=1)
print("Saved train/val/test splits + category_maps to data/processed/")

Saved train/val/test splits + category_maps to data/processed/


## Next: `04_baseline_and_models.ipynb`

**Status of this notebook: IMPLEMENTED.** Feature build, split, and the leakage self-check above all ran for
real (see `PROGRESS_LEDGER.md`).